# Looking At Galaxies and PSFs in the COSMOS Deep Field in DP2

This notebook explores some of what is possible in the deep fields with full coverage in DP2.

For rapidly getting up to speed, the catalog queried from the server has already been made available, but the full query is also demonstrated.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
import astropy.units as units
from lsst.rsp import get_tap_service, retrieve_query
from astropy.coordinates import SkyCoord
import skyproj
import hpgeom
import h5py

In [ ]:
service = get_tap_service("tap")

In [ ]:
columns = [
    "coord_ra",
    "coord_raErr",
    "coord_dec",
    "coord_decErr",
    "u_psfFlux",
    "g_psfFlux",
    "r_psfFlux",
    "i_psfFlux",
    "z_psfFlux",
    "y_psfFlux",
    "u_psfFluxErr",
    "g_psfFluxErr",
    "r_psfFluxErr",
    "i_psfFluxErr",
    "z_psfFluxErr",
    "y_psfFluxErr",
    "u_sersicFlux",
    "g_sersicFlux",
    "r_sersicFlux",
    "i_sersicFlux",
    "z_sersicFlux",
    "y_sersicFlux",
    "u_sersicFluxErr",
    "g_sersicFluxErr",
    "r_sersicFluxErr",
    "i_sersicFluxErr",
    "z_sersicFluxErr",
    "y_sersicFluxErr",
    "u_cModelFlux",
    "g_cModelFlux",
    "r_cModelFlux",
    "i_cModelFlux",
    "z_cModelFlux",
    "y_cModelFlux",
    "u_cModelFluxErr",
    "g_cModelFluxErr",
    "r_cModelFluxErr",
    "i_cModelFluxErr",
    "z_cModelFluxErr",
    "y_cModelFluxErr",
    "u_cModel_flag",
    "g_cModel_flag",
    "r_cModel_flag",
    "i_cModel_flag",
    "z_cModel_flag",
    "y_cModel_flag",
    "u_blendedness_flag",
    "g_blendedness_flag",
    "r_blendedness_flag",
    "i_blendedness_flag",
    "z_blendedness_flag",
    "y_blendedness_flag",
    "u_pixelFlags_bad",
    "g_pixelFlags_bad",
    "r_pixelFlags_bad",
    "i_pixelFlags_bad",
    "z_pixelFlags_bad",
    "y_pixelFlags_bad",
    "u_pixelFlags_inexact_psf",
    "g_pixelFlags_inexact_psf",
    "r_pixelFlags_inexact_psf",
    "i_pixelFlags_inexact_psf",
    "z_pixelFlags_inexact_psf",
    "y_pixelFlags_inexact_psf",
    "u_pixelFlags_nodata",
    "g_pixelFlags_nodata",
    "r_pixelFlags_nodata",
    "i_pixelFlags_nodata",
    "z_pixelFlags_nodata",
    "y_pixelFlags_nodata",
    "u_invalidPsfFlag",
    "g_invalidPsfFlag",
    "r_invalidPsfFlag",
    "i_invalidPsfFlag",
    "z_invalidPsfFlag",
    "y_invalidPsfFlag",
    "u_centroid_flag",
    "g_centroid_flag",
    "r_centroid_flag",
    "i_centroid_flag",
    "z_centroid_flag",
    "y_centroid_flag",
    "parentObjectId",
    "refBand",
    "refExtendedness",
    "ebv",
    "detect_isIsolated",
    "detect_fromBlend",
    "detect_isDeblendedModelSource",
    "g_epoch",
    "u_epoch",
    "r_epoch",
    "i_epoch",
    "z_epoch",
    "y_epoch",
    "objectId",
    "griz_model_extendedness",
    "u_ixx",
    "u_iyy",
    "u_ixy",
    "u_ixxPSF",
    "u_iyyPSF",
    "u_ixyPSF",
    "g_ixx",
    "g_iyy",
    "g_ixy",
    "g_ixxPSF",
    "g_iyyPSF",
    "g_ixyPSF",
    "r_ixx",
    "r_iyy",
    "r_ixy",
    "r_ixxPSF",
    "r_iyyPSF",
    "r_ixyPSF",
    "i_ixx",
    "i_iyy",
    "i_ixy",
    "i_ixxPSF",
    "i_iyyPSF",
    "i_ixyPSF",
    "z_ixx",
    "z_iyy",
    "z_ixy",
    "z_ixxPSF",
    "z_iyyPSF",
    "z_ixyPSF",
    "y_ixx",
    "y_iyy",
    "y_ixy",
    "y_ixxPSF",
    "y_iyyPSF",
    "y_ixyPSF",
]
column_string = ",".join(columns)

In [ ]:
# Set do_query = True if you want to execute the full query

results = None
do_query = False

In [ ]:
if do_query:
    query = f"SELECT {column_string} " \
            "FROM dp2.Object " \
            "WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), " \
            "CIRCLE('ICRS', 150.11916667, 2.20583333, 1.0)) = 1"
    job = service.submit_job(query)
    print('Job URL is', job.url)
    print('Pre-Job phase is', job.phase)
    job.run()
    job.wait(phases=['COMPLETED', 'ERROR'])
    print('Job phase is', job.phase)
    results = job.fetch_result().to_table()
    print("Done fetching results")

In [ ]:
if results is None:
    results = Table.read("/home/erykoff/data/cosmos_object_selection.fits")
print("Loaded ", len(results), " rows.")
print(results)

In [ ]:
# Select out objects that have valid PSFs in all of ugrizy.

use = (
    (results["u_psfFlux"] > 0)
    & (~results["u_invalidPsfFlag"])
    & (results["g_psfFlux"] > 0)
    & (~results["g_invalidPsfFlag"])
    & (results["r_psfFlux"] > 0)
    & (~results["r_invalidPsfFlag"])
    & (results["i_psfFlux"] > 0)
    & (~results["i_invalidPsfFlag"])
    & (results["z_psfFlux"] > 0)
    & (~results["z_invalidPsfFlag"])
    & (results["y_psfFlux"] > 0)
    & (~results["y_invalidPsfFlag"])
)
print(use.sum())


In [ ]:
# Compute colors from the PSF fluxes for exploration.

umg = results["u_psfFlux"][use].quantity.to_value(units.ABmag) - results["g_psfFlux"][use].quantity.to_value(units.ABmag)
gmr = results["g_psfFlux"][use].quantity.to_value(units.ABmag) - results["r_psfFlux"][use].quantity.to_value(units.ABmag)
rmi = results["r_psfFlux"][use].quantity.to_value(units.ABmag) - results["i_psfFlux"][use].quantity.to_value(units.ABmag)
imz = results["i_psfFlux"][use].quantity.to_value(units.ABmag) - results["z_psfFlux"][use].quantity.to_value(units.ABmag)
zmy = results["z_psfFlux"][use].quantity.to_value(units.ABmag) - results["y_psfFlux"][use].quantity.to_value(units.ABmag)

imag = results["i_psfFlux"][use].quantity.to_value(units.ABmag)
imag_err = 1.086*results["i_psfFluxErr"][use] / results["i_psfFlux"][use]

In [ ]:
# Look at the color/color plots, splitting star/galaxies with refExtendedness (default)

bright_objects = (imag_err < 0.05)

plt.hexbin(rmi[bright_objects], gmr[bright_objects], bins="log", extent=[-0.3, 1.6, -0.2, 1.7])
plt.colorbar(label="Density")
plt.xlabel("r - i")
plt.ylabel("g - r")
plt.show()

stars = results["refExtendedness"][use][bright_objects] < 0.5
galaxies = results["refExtendedness"][use][bright_objects] > 0.5

plt.hexbin(rmi[bright_objects][stars], gmr[bright_objects][stars], bins="log", extent=[-0.3, 1.6, -0.2, 1.7])
plt.colorbar(label="Density")
plt.xlabel("r - i")
plt.ylabel("g - r")
plt.title("Stars")
plt.show()

plt.hexbin(rmi[bright_objects][galaxies], gmr[bright_objects][galaxies], bins="log", extent=[-0.3, 1.6, -0.2, 1.7])
plt.colorbar(label="Density")
plt.xlabel("r - i")
plt.ylabel("g - r")
plt.title("Galaxies")
plt.show()


Note the effective split of stars and galaxies for the bright objects.  Also note how the stellar locus comes close to the galaxies at r-i ~ 0.4 and g-r ~ 0.8.  This is where the red galaxies and the stars have similar colors and is a nuisance for cluster finding around redshift ~0.7.

In [ ]:
# Looking at color/color with different colors

bright_objects = (imag_err < 0.05)

plt.hexbin(gmr[bright_objects], umg[bright_objects], bins="log", extent=[-0.3, 1.6, -0.2, 3.0])
plt.colorbar(label="Density")
plt.xlabel("g - r")
plt.ylabel("u - g")
plt.show()

stars = results["refExtendedness"][use][bright_objects] < 0.5
galaxies = results["refExtendedness"][use][bright_objects] > 0.5

plt.hexbin(gmr[bright_objects][stars], umg[bright_objects][stars], bins="log", extent=[-0.3, 1.6, -0.2, 3.0])
plt.colorbar(label="Density")
plt.xlabel("g - r")
plt.ylabel("u - g")
plt.title("Stars")
plt.show()

plt.hexbin(gmr[bright_objects][galaxies], umg[bright_objects][galaxies], bins="log", extent=[-0.3, 1.6, -0.2, 3.0])
plt.colorbar(label="Density")
plt.xlabel("g - r")
plt.ylabel("u - g")
plt.title("Galaxies")
plt.show()

# Galaxy Redshift Evolution

In [ ]:
FILENAME = "/home/erykoff/data/CWWSB_LSST_Roman_colors_YJH.hdf5"

BAND_LIST = [
'u',
'g',
'r',
'i',
'z',
'y',
]

# (Beńitez 2004a)
SED_list = [
    "El_B2004a", 
    #"Im_B2004a",
    #"SB2_B2004a",
    #"SB3_B2004a",
    "Sbc_B2004a",
    "Scd_B2004a",
    #"ssp_5Myr_z008",
    #"ssp_25Myr_z008",
]

def get_redshift():
    with h5py.File(FILENAME, "r") as hf:
        redshift = hf["redshift"][:]
    return redshift

def get_color(SED_name, band1_name, band2_name):
    
    band1_index = BAND_LIST.index(band1_name)
    band2_index = BAND_LIST.index(band2_name)

    color_name = f"{band1_name}{band2_name}"
    
    with h5py.File(FILENAME, "r") as hf:
        if (band1_index+1)==band2_index:
            color = hf[f"{SED_name}_{color_name}"][:]
        else:
            color = np.zeros_like(hf["redshift"][:])
            
            band1_name_tmp = band1_name
            while 1:
                band2_name_tmp = BAND_LIST[BAND_LIST.index(band1_name_tmp)+1]
                color_name_tmp = f"{band1_name_tmp}{band2_name_tmp}"
                color_tmp = hf[f"{SED_name}_{color_name_tmp}"][:]
                color += color_tmp
                band1_name_tmp = band2_name_tmp
                if band2_name_tmp == band2_name:
                    break
            
    return color


In [ ]:
def make_cc_plot2(ax, SED_name, band1_name, band2_name, band3_name, band4_name, color='C0'):

    alpha = 1
    redshift = get_redshift()
    sel = redshift <= 2.5
    
    color1 = get_color(SED_name, band1_name, band2_name)
    color2 = get_color(SED_name, band3_name, band4_name)

    def get_name(s):
        """Return the part of s before the first underscore."""
        return s.split('_', 1)[0]
    
    ax.plot(color1[sel], color2[sel], label=get_name(SED_name), linewidth=1, alpha=alpha, color=color)

    # 'o', 's', '^', 'D', 'X', 'v'
    # 'o', '>', 's', '*', 'P', 'X'
    sel = redshift==0
    ax.plot(color1[sel], color2[sel], marker='o', markersize=5,  linewidth=0, color=color, alpha=alpha)

    sel = redshift==0.5
    ax.plot(color1[sel], color2[sel], marker='D', markersize=5, linewidth=0, color=color, alpha=alpha)

    sel = redshift==1.
    ax.plot(color1[sel], color2[sel], marker='s', markersize=5, linewidth=0, color=color, alpha=alpha)

    sel = redshift==1.5
    ax.plot(color1[sel], color2[sel], marker='*', markersize=7, linewidth=0, color=color, alpha=alpha)

    sel = redshift==2.0
    ax.plot(color1[sel], color2[sel], marker='+', markersize=7, linewidth=0, color=color, alpha=alpha)

    sel = redshift==2.5
    ax.plot(color1[sel], color2[sel], marker='x', markersize=7, linewidth=0, color=color, alpha=alpha)
    
    return 0


In [ ]:
# We can load in a library of SEDs and follow their tracks in color-color space

# At first we are using the psf fluxes which are optimized for points but actually get decent colors for galaxies.

plt.hexbin(
    rmi[bright_objects][galaxies], 
    gmr[bright_objects][galaxies], 
    bins="log", 
    extent=[-0.3, 1.6, -0.2, 1.7],
    cmap="binary_r",
)
make_cc_plot2(plt.gca(), "El_B2004a", "r", "i", "g", "r", color="red")
make_cc_plot2(plt.gca(), "Sbc_B2004a", "r", "i", "g", "r", color="magenta")
make_cc_plot2(plt.gca(), "Scd_B2004a", "r", "i", "g", "r", color="blue")

plt.colorbar(label="Density")
plt.xlabel("r - i")
plt.ylabel("g - r")
plt.title("Galaxies")
plt.legend()
plt.show()


In [ ]:
umg_sersic = results["u_sersicFlux"][use].quantity.to_value(units.ABmag) - results["g_sersicFlux"][use].quantity.to_value(units.ABmag)
gmr_sersic = results["g_sersicFlux"][use].quantity.to_value(units.ABmag) - results["r_sersicFlux"][use].quantity.to_value(units.ABmag)
rmi_sersic = results["r_sersicFlux"][use].quantity.to_value(units.ABmag) - results["i_sersicFlux"][use].quantity.to_value(units.ABmag)
imz_sersic = results["i_sersicFlux"][use].quantity.to_value(units.ABmag) - results["z_sersicFlux"][use].quantity.to_value(units.ABmag)
zmy_sersic = results["z_sersicFlux"][use].quantity.to_value(units.ABmag) - results["y_sersicFlux"][use].quantity.to_value(units.ABmag)

imag_sersic = results["i_sersicFlux"][use].quantity.to_value(units.ABmag)
imag_sersic_err = 1.086*results["i_sersicFluxErr"][use] / results["i_sersicFlux"][use]

gals = (imag_err < 0.1) & (results["refExtendedness"][use] > 0.5)


In [ ]:
# Repeat the same plots with Sersic model fluxes.

plt.hexbin(
    rmi_sersic[gals], 
    gmr_sersic[gals], 
    bins="log", 
    extent=[-0.3, 1.6, -0.2, 1.7],
    cmap="binary_r",
)
make_cc_plot2(plt.gca(), "El_B2004a", "r", "i", "g", "r", color="red")
make_cc_plot2(plt.gca(), "Sbc_B2004a", "r", "i", "g", "r", color="magenta")
make_cc_plot2(plt.gca(), "Scd_B2004a", "r", "i", "g", "r", color="blue")

plt.colorbar(label="Density")
plt.xlabel("r - i")
plt.ylabel("g - r")
plt.title("Galaxies")
plt.legend()
plt.show()


# Psf Sizes And Biases

In this section we are going to look at how well the PSF models describe the star PSFs in terms of size T (which is sigma^2).

In [ ]:
all_stars = (results["refExtendedness"][use] < 0.5)
imag = results[f"i_psfFlux"][use].quantity.to_value(units.ABmag)[all_stars]

for band in ["g", "r", "i"]:    
    star_T = results[f"{band}_ixx"][use][all_stars] + results[f"{band}_iyy"][use][all_stars]
    psf_T = results[f"{band}_ixxPSF"][use][all_stars] + results[f"{band}_iyyPSF"][use][all_stars]
    mag = results[f"{band}_psfFlux"][use].quantity.to_value(units.ABmag)[all_stars]

    dT_over_T = (star_T - psf_T) / star_T

    good = (np.isfinite(star_T) & np.isfinite(psf_T) & (imag < 24) & (np.abs(dT_over_T) < 0.5))

    y_lo, y_hi = np.nanpercentile(dT_over_T[good], [1, 99])
    x_lo, x_hi = np.nanpercentile(mag[good], [1, 99])

    plt.hexbin(mag[good], dT_over_T[good], bins="log", extent=[x_lo, x_hi, y_lo, y_hi])
    plt.xlabel(f"{band} mag")
    plt.ylabel("dT / T")
    plt.title(f"{band}")
    plt.show()

The size bias looks good overall, and fortunately is flat as a function magnitude/flux.  This means the overall background is reasonably well subtracted.

However, something is fishy for the g band ...

In [ ]:
# The same plots, but now split by red and blue stars.

all_stars = (results["refExtendedness"][use] < 0.5)
gmag = results[f"g_psfFlux"][use].quantity.to_value(units.ABmag)[all_stars]
imag = results[f"i_psfFlux"][use].quantity.to_value(units.ABmag)[all_stars]
gmi = gmag - imag

for band in ["g", "r", "i"]:
    star_T = results[f"{band}_ixx"][use][all_stars] + results[f"{band}_iyy"][use][all_stars]
    psf_T = results[f"{band}_ixxPSF"][use][all_stars] + results[f"{band}_iyyPSF"][use][all_stars]
    mag = results[f"{band}_psfFlux"][use].quantity.to_value(units.ABmag)[all_stars]

    dT_over_T = (star_T - psf_T) / star_T

    good = (np.isfinite(star_T) & np.isfinite(psf_T) & (imag < 24) & (np.abs(dT_over_T) < 0.5))

    y_lo, y_hi = np.nanpercentile(dT_over_T[good], [1, 99])
    x_lo, x_hi = np.nanpercentile(mag[good], [1, 99])

    med_gmi = np.median(gmi[good])
    blue = (gmi[good] < med_gmi)
    red = (gmi[good] >= med_gmi)

    plt.hexbin(mag[good][blue], dT_over_T[good][blue], bins="log", extent=[x_lo, x_hi, y_lo, y_hi], cmap="Blues", alpha=0.5)
    plt.hexbin(mag[good][red], dT_over_T[good][red], bins="log", extent=[x_lo, x_hi, y_lo, y_hi], cmap="Reds", alpha=0.5)
    plt.xlabel(f"{band} mag")
    plt.ylabel("dT / T")
    plt.title(f"{band}, split colors")
    plt.show()


Now isn't that interesting!  It seems that the PSF size bias (and the PSF size) depends on the star color.  This is the "chromatic seeing" such that bluer light has a broader PSF than redder light, and fattens the PSF of bluer stars which have more blue light.

DR1 will have a chromatic term in the PSF model to help with this.

How do these vary over the COSMOS footprint?

(This isn't very interesting, but can you find a quantity that is interesting?)

In [ ]:
all_stars = (results["refExtendedness"][use] < 0.5)
imag = np.asarray(results[f"i_psfFlux"][use].quantity.to_value(units.ABmag)[all_stars])
ra = np.asarray(results["coord_ra"][use][all_stars])
dec = np.asarray(results["coord_dec"][use][all_stars])
lon0 = np.median(ra)

for band in ["g", "r", "i"]:    
    star_T = np.asarray(results[f"{band}_ixx"][use][all_stars] + results[f"{band}_iyy"][use][all_stars])
    psf_T = np.asarray(results[f"{band}_ixxPSF"][use][all_stars] + results[f"{band}_iyyPSF"][use][all_stars])
    mag = np.asarray(results[f"{band}_psfFlux"][use].quantity.to_value(units.ABmag)[all_stars])

    dT_over_T = (star_T - psf_T) / star_T

    # Look at brighter stars.
    good = (np.isfinite(star_T) & np.isfinite(psf_T) & (imag < 22) & (np.abs(dT_over_T) < 0.5))

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111)

    sp = skyproj.McBrydeSkyproj(ax=ax, lon0=lon0)
    sp.draw_hpxbin(
        ra[good],
        dec[good],
        C=dT_over_T[good],
        nest=True,
        nside=1024,
    )
    sp.draw_colorbar(label=f"{band} dT / T")
    plt.show()


# Homework Exercise

Make a plot of the stellar locus, but instead of color-coded by density, do the color-coding by magnitude.

What does this tell us about the differences between red and blue stars?  Does this have any implications for the interpretation of systematic biases?